# Incident Response Runbook: Impact - Data Encrypted for Impact

**Tactic:** Impact
**Technique:** Data Encrypted for Impact (T1486)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Data Encrypted for Impact activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Data Encrypted for Impact (Ransomware) - T1486',
    'description': 'Potential ransomware attack detected - data encryption activities identified',
    'severity': 'CRITICAL',
    'category': 'Malware',
    'tags': ['ransomware', 'data-encryption', 'impact', 't1486']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Detection queries for ransomware indicators
print("\n Executing ransomware detection queries...")

# Splunk query for file encryption patterns
encryption_query = """
index=* (sourcetype=fs_notification OR sourcetype=windows_security)
| eval file_ext = lower(replace(_raw, ".*\.([^\.]+)$", "$1"))
| where file_ext IN ("encrypted", "locked", "crypto", "ransom", "enc", "aes", "rsa")
| stats count by host, file_path, file_ext, user
| where count > 10
| sort -count
"""

encryption_results = splunk.execute_query(encryption_query)
print(f" Found {len(encryption_results)} potential encryption patterns")

# Splunk query for ransomware note files
ransom_note_query = """
index=* sourcetype=fs_notification
| regex _raw="(?i)(readme|how_to_decrypt|decrypt_instructions|your_files_are_encrypted|pay_bitcoin)"
| stats count by host, file_path
| sort -count
"""

ransom_results = splunk.execute_query(ransom_note_query)
print(f" Found {len(ransom_results)} potential ransom note files")

# CrowdStrike query for suspicious process behavior
cs_query = """
event_simpleName=ProcessRollup2
| where (ImageFileName =~ "*vssadmin*" OR ImageFileName =~ "*wbadmin*" OR ImageFileName =~ "*bcdedit*")
OR (CommandLine =~ "*delete shadows*" OR CommandLine =~ "*shadowcopy*" OR CommandLine =~ "*recovery*")
| stats count by ComputerName, ImageFileName, CommandLine
"""

cs_results = crowdstrike.execute_query(cs_query)
print(f" Found {len(cs_results)} suspicious process activities")

# Check for known ransomware IOCs in MISP
ransomware_indicators = misp.search_indicators("ransomware", limit=50)
print(f" Retrieved {len(ransomware_indicators)} ransomware indicators from MISP")

# Compile detection results
detection_summary = {
    'case_id': case_id,
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Impact',
    'technique': 'Data Encrypted for Impact (T1486)',
    'severity': 'CRITICAL',
    'indicators': {
        'file_encryption_patterns': len(encryption_results),
        'ransom_notes': len(ransom_results),
        'suspicious_processes': len(cs_results),
        'misp_indicators': len(ransomware_indicators)
    },
    'affected_hosts': list(set([r.get('host', r.get('ComputerName', 'unknown')) 
                               for r in encryption_results + cs_results]))
}

print(f"\n📊 Detection Summary:")
print(f"  Case ID: {detection_summary['case_id']}")
print(f"  Severity: {detection_summary['severity']}")
print(f"  Affected Hosts: {len(detection_summary['affected_hosts'])}")
print(f"  Encryption Patterns: {detection_summary['indicators']['file_encryption_patterns']}")
print(f"  Ransom Notes: {detection_summary['indicators']['ransom_notes']}")
print(f"  Suspicious Processes: {detection_summary['indicators']['suspicious_processes']}")

# Trigger Shuffle workflow for automated response
if detection_summary['indicators']['file_encryption_patterns'] > 0:
    shuffle.trigger_workflow('ransomware_detection_response', detection_summary)
    print(" Triggered automated ransomware response workflow")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []

# Isolate affected systems via CrowdStrike
print("\n Isolating affected systems...")
for host in detection_summary['affected_hosts']:
    try:
        isolation_result = crowdstrike.isolate_host(host)
        if isolation_result.get('success'):
            containment_actions.append(f" Isolated host: {host}")
            print(f"   Isolated host: {host}")
        else:
            containment_actions.append(f" Failed to isolate host: {host}")
            print(f"   Failed to isolate host: {host}")
    except Exception as e:
        containment_actions.append(f" Error isolating {host}: {str(e)}")
        print(f"   Error isolating {host}: {str(e)}")

# Block suspicious network connections
print("\n Blocking suspicious network connections...")
network_block_query = """
index=* sourcetype=network_traffic
| where dest_ip IN (ransomware_indicators{:dest_ip})
| stats count by src_ip, dest_ip, dest_port
| where count > 5
"""

network_results = splunk.execute_query(network_block_query)
for result in network_results:
    try:
        block_result = shuffle.block_ip(result['dest_ip'])
        if block_result.get('success'):
            containment_actions.append(f" Blocked IP: {result['dest_ip']}")
            print(f"   Blocked IP: {result['dest_ip']}")
    except Exception as e:
        containment_actions.append(f" Failed to block IP {result['dest_ip']}: {str(e)}")
        print(f"   Failed to block IP {result['dest_ip']}: {str(e)}")

# Disable compromised accounts
print("\n👤 Disabling compromised accounts...")
account_query = """
index=* sourcetype=authentication
| where user IN (detection_summary['affected_hosts']{:user})
| where action="failed_login" OR action="suspicious_activity"
| stats count by user, src_ip
| where count > 3
"""

account_results = splunk.execute_query(account_query)
for result in account_results:
    try:
        disable_result = shuffle.disable_account(result['user'])
        if disable_result.get('success'):
            containment_actions.append(f" Disabled account: {result['user']}")
            print(f"   Disabled account: {result['user']}")
    except Exception as e:
        containment_actions.append(f" Failed to disable account {result['user']}: {str(e)}")
        print(f"   Failed to disable account {result['user']}: {str(e)}")

# Enable enhanced monitoring
print("\n📊 Enabling enhanced monitoring...")
try:
    monitoring_config = {
        'ransomware_detection': True,
        'file_system_monitoring': True,
        'network_anomaly_detection': True,
        'process_behavior_analysis': True
    }
    splunk.update_monitoring_rules(monitoring_config)
    containment_actions.append(" Enhanced monitoring enabled")
    print("   Enhanced monitoring enabled")
except Exception as e:
    containment_actions.append(f" Failed to enable enhanced monitoring: {str(e)}")
    print(f"   Failed to enable enhanced monitoring: {str(e)}")

# Update IRIS case with containment actions
iris.update_case(case_id, {
    'containment_actions': containment_actions,
    'containment_timestamp': datetime.now().isoformat(),
    'status': 'containment_in_progress'
})

print(f"\n Containment Summary:")
print(f"  Actions Taken: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in containment_actions if a.startswith('')])}")

# Trigger Shuffle workflow for containment verification
shuffle.trigger_workflow('containment_verification', {
    'case_id': case_id,
    'actions_taken': containment_actions
})
print(" Triggered containment verification workflow")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []

# Terminate malicious processes via CrowdStrike
print("\n💀 Terminating malicious processes...")
process_query = """
event_simpleName=ProcessRollup2
| where ImageFileName =~ "*ransom*" OR ImageFileName =~ "*encrypt*" OR ImageFileName =~ "*crypto*"
OR CommandLine =~ "*encrypt*" OR CommandLine =~ "*ransom*"
| stats count by ComputerName, ProcessId, ImageFileName
"""

malicious_processes = crowdstrike.execute_query(process_query)
for process in malicious_processes:
    try:
        kill_result = crowdstrike.terminate_process(process['ComputerName'], process['ProcessId'])
        if kill_result.get('success'):
            eradication_actions.append(f" Terminated process: {process['ImageFileName']} on {process['ComputerName']}")
            print(f"   Terminated process: {process['ImageFileName']} on {process['ComputerName']}")
    except Exception as e:
        eradication_actions.append(f" Failed to terminate process {process['ProcessId']}: {str(e)}")
        print(f"   Failed to terminate process {process['ProcessId']}: {str(e)}")

# Remove ransomware files and ransom notes
print("\n🗑 Removing ransomware files...")
file_removal_query = """
index=* sourcetype=fs_notification
| where file_path =~ "*readme*.txt" OR file_path =~ "*decrypt*.txt" OR file_path =~ "*instructions*.txt"
OR file_path =~ "*.encrypted" OR file_path =~ "*.locked" OR file_path =~ "*.ransom"
| stats count by host, file_path
"""

ransomware_files = splunk.execute_query(file_removal_query)
for file_info in ransomware_files:
    try:
        removal_result = crowdstrike.remove_file(file_info['host'], file_info['file_path'])
        if removal_result.get('success'):
            eradication_actions.append(f" Removed file: {file_info['file_path']} from {file_info['host']}")
            print(f"   Removed file: {file_info['file_path']} from {file_info['host']}")
    except Exception as e:
        eradication_actions.append(f" Failed to remove file {file_info['file_path']}: {str(e)}")
        print(f"   Failed to remove file {file_info['file_path']}: {str(e)}")

# Reset compromised credentials
print("\n🔑 Resetting compromised credentials...")
credential_query = """
index=* sourcetype=authentication
| where user IN (detection_summary['affected_hosts']{:user})
| where action="successful_login" AND risk_score > 80
| stats count by user, src_ip, timestamp
| where count > 1
"""

compromised_accounts = splunk.execute_query(credential_query)
for account in compromised_accounts:
    try:
        reset_result = shuffle.reset_password(account['user'])
        if reset_result.get('success'):
            eradication_actions.append(f" Reset password for: {account['user']}")
            print(f"   Reset password for: {account['user']}")
    except Exception as e:
        eradication_actions.append(f" Failed to reset password for {account['user']}: {str(e)}")
        print(f"   Failed to reset password for {account['user']}: {str(e)}")

# Clear system caches and temporary files
print("\n Clearing system caches...")
for host in detection_summary['affected_hosts']:
    try:
        cache_clear_result = crowdstrike.clear_system_cache(host)
        if cache_clear_result.get('success'):
            eradication_actions.append(f" Cleared system cache on: {host}")
            print(f"   Cleared system cache on: {host}")
    except Exception as e:
        eradication_actions.append(f" Failed to clear cache on {host}: {str(e)}")
        print(f"   Failed to clear cache on {host}: {str(e)}")

# Verify threat removal
print("\n✅ Verifying threat removal...")
verification_results = []
for host in detection_summary['affected_hosts']:
    try:
        scan_result = crowdstrike.perform_threat_scan(host)
        if scan_result.get('threats_found', 0) == 0:
            verification_results.append(f" Clean scan on {host}")
            print(f"   Clean scan on {host}")
        else:
            verification_results.append(f" Threats still present on {host}: {scan_result.get('threats_found')}")
            print(f"   Threats still present on {host}: {scan_result.get('threats_found')}")
    except Exception as e:
        verification_results.append(f" Scan failed on {host}: {str(e)}")
        print(f"   Scan failed on {host}: {str(e)}")

# Update IRIS case with eradication actions
iris.update_case(case_id, {
    'eradication_actions': eradication_actions,
    'verification_results': verification_results,
    'eradication_timestamp': datetime.now().isoformat(),
    'status': 'eradication_complete'
})

print(f"\n Eradication Summary:")
print(f"  Actions Taken: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Verification Results: {len([r for r in verification_results if r.startswith('')])} clean hosts")

# Share indicators with MISP
if len(ransomware_indicators) > 0:
    misp.share_indicators(ransomware_indicators, case_id)
    print(" Shared indicators with MISP community")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []

# Re-enable isolated systems
print("\n🔓 Re-enabling isolated systems...")
for host in detection_summary['affected_hosts']:
    try:
        reenable_result = crowdstrike.reenable_host(host)
        if reenable_result.get('success'):
            recovery_actions.append(f" Re-enabled host: {host}")
            print(f"   Re-enabled host: {host}")
        else:
            recovery_actions.append(f" Failed to re-enable host: {host}")
            print(f"   Failed to re-enable host: {host}")
    except Exception as e:
        recovery_actions.append(f" Error re-enabling {host}: {str(e)}")
        print(f"   Error re-enabling {host}: {str(e)}")

# Restore accounts
print("\n👤 Restoring user accounts...")
for account in compromised_accounts:
    try:
        restore_result = shuffle.restore_account(account['user'])
        if restore_result.get('success'):
            recovery_actions.append(f" Restored account: {account['user']}")
            print(f"   Restored account: {account['user']}")
    except Exception as e:
        recovery_actions.append(f" Failed to restore account {account['user']}: {str(e)}")
        print(f"   Failed to restore account {account['user']}: {str(e)}")

# Validate system integrity
print("\n Validating system integrity...")
integrity_checks = []
for host in detection_summary['affected_hosts']:
    try:
        integrity_result = crowdstrike.validate_system_integrity(host)
        if integrity_result.get('integrity_valid'):
            integrity_checks.append(f" System integrity valid on {host}")
            print(f"   System integrity valid on {host}")
        else:
            integrity_checks.append(f" Integrity issues on {host}: {integrity_result.get('issues', [])}")
            print(f"   Integrity issues on {host}: {integrity_result.get('issues', [])}")
    except Exception as e:
        integrity_checks.append(f" Integrity check failed on {host}: {str(e)}")
        print(f"   Integrity check failed on {host}: {str(e)}")

# Restore from backups (if available)
print("\n💾 Checking for backup restoration...")
backup_query = """
index=* sourcetype=backup_logs
| where host IN (detection_summary['affected_hosts'])
| where status="success"
| stats latest(timestamp) as last_backup by host, backup_type
"""

backup_results = splunk.execute_query(backup_query)
for backup in backup_results:
    try:
        # Check if backup is recent enough (within last 24 hours)
        backup_time = datetime.fromisoformat(backup['last_backup'])
        if datetime.now() - backup_time < timedelta(hours=24):
            restore_result = shuffle.restore_from_backup(backup['host'], backup['backup_type'])
            if restore_result.get('success'):
                recovery_actions.append(f" Restored from backup: {backup['host']}")
                print(f"   Restored from backup: {backup['host']}")
            else:
                recovery_actions.append(f" Backup restoration failed for {backup['host']}")
                print(f"   Backup restoration failed for {backup['host']}")
        else:
            recovery_actions.append(f" Backup too old for {backup['host']}: {backup['last_backup']}")
            print(f"   Backup too old for {backup['host']}: {backup['last_backup']}")
    except Exception as e:
        recovery_actions.append(f" Backup check failed for {backup['host']}: {str(e)}")
        print(f"   Backup check failed for {backup['host']}: {str(e)}")

# Monitor for recurrence
print("\n Establishing monitoring for recurrence...")
try:
    recurrence_monitoring = {
        'hosts': detection_summary['affected_hosts'],
        'duration_days': 30,
        'alert_threshold': 'low',
        'indicators': ['ransomware_patterns', 'encryption_activity', 'suspicious_processes']
    }
    splunk.setup_recurrence_monitoring(recurrence_monitoring)
    recovery_actions.append(" Recurrence monitoring established")
    print("   Recurrence monitoring established")
except Exception as e:
    recovery_actions.append(f" Failed to establish recurrence monitoring: {str(e)}")
    print(f"   Failed to establish recurrence monitoring: {str(e)}")

# Update IRIS case with recovery actions
iris.update_case(case_id, {
    'recovery_actions': recovery_actions,
    'integrity_checks': integrity_checks,
    'recovery_timestamp': datetime.now().isoformat(),
    'status': 'recovery_complete'
})

print(f"\n Recovery Summary:")
print(f"  Actions Taken: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Integrity Checks Passed: {len([c for c in integrity_checks if c.startswith('')])}")

# Trigger Shuffle workflow for recovery verification
shuffle.trigger_workflow('recovery_verification', {
    'case_id': case_id,
    'actions_taken': recovery_actions,
    'integrity_checks': integrity_checks
})
print(" Triggered recovery verification workflow")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []

# Generate incident report
print("\n Generating incident report...")
try:
    incident_report = {
        'case_id': case_id,
        'title': 'Ransomware Incident Response Report',
        'timeline': {
            'detection': detection_summary['timestamp'],
            'containment': datetime.now().isoformat(),  # Would be actual timestamp
            'eradication': datetime.now().isoformat(),   # Would be actual timestamp
            'recovery': datetime.now().isoformat(),      # Would be actual timestamp
            'closure': datetime.now().isoformat()
        },
        'impact_assessment': {
            'affected_hosts': len(detection_summary['affected_hosts']),
            'data_encrypted': detection_summary['indicators']['file_encryption_patterns'],
            'business_impact': 'HIGH',  # Would be determined by business impact analysis
            'ransom_paid': False,
            'data_recovered': len([a for a in recovery_actions if 'backup' in a.lower()])
        },
        'response_metrics': {
            'detection_to_containment': 'TBD',  # Would calculate actual time
            'containment_to_eradication': 'TBD',
            'total_resolution_time': 'TBD',
            'automation_level': 'HIGH'
        },
        'lessons_learned': [
            'Enhanced file system monitoring detected encryption patterns early',
            'Automated isolation prevented lateral movement',
            'Backup restoration was successful for critical systems',
            'MISP integration provided valuable threat intelligence'
        ]
    }

    report_id = iris.generate_report(case_id, incident_report)
    post_incident_actions.append(f" Generated incident report: {report_id}")
    print(f"   Generated incident report: {report_id}")

except Exception as e:
    post_incident_actions.append(f" Failed to generate report: {str(e)}")
    print(f"   Failed to generate report: {str(e)}")

# Document lessons learned
print("\n Documenting lessons learned...")
lessons_learned = [
    "Implement real-time file encryption monitoring",
    "Regular backup testing and validation",
    "Employee training on ransomware awareness",
    "Network segmentation to limit lateral movement",
    "Automated response workflows reduce response time",
    "Integration with threat intelligence platforms improves detection"
]

try:
    iris.add_lessons_learned(case_id, lessons_learned)
    post_incident_actions.append(" Documented lessons learned")
    print("   Documented lessons learned")
except Exception as e:
    post_incident_actions.append(f" Failed to document lessons learned: {str(e)}")
    print(f"   Failed to document lessons learned: {str(e)}")

# Implement preventive measures
print("\n Implementing preventive measures...")
preventive_measures = []

try:
    # Update security policies
    policy_updates = {
        'ransomware_protection': True,
        'backup_frequency': 'daily',
        'endpoint_protection': 'enhanced',
        'network_segmentation': 'strict'
    }
    shuffle.update_security_policies(policy_updates)
    preventive_measures.append(" Updated security policies")
    print("   Updated security policies")

    # Enhance monitoring rules
    monitoring_updates = {
        'file_encryption_detection': True,
        'ransomware_behavior_analysis': True,
        'automated_quarantine': True
    }
    splunk.update_monitoring_rules(monitoring_updates)
    preventive_measures.append(" Enhanced monitoring rules")
    print("   Enhanced monitoring rules")

    # Update threat intelligence feeds
    misp.update_feeds(['ransomware', 'file_encryptors'])
    preventive_measures.append(" Updated threat intelligence feeds")
    print("   Updated threat intelligence feeds")

except Exception as e:
    preventive_measures.append(f" Failed to implement preventive measures: {str(e)}")
    print(f"   Failed to implement preventive measures: {str(e)}")

# Conduct post-incident review
print("\n Conducting post-incident review...")
try:
    review_findings = {
        'effectiveness_rating': 'HIGH',
        'areas_for_improvement': [
            'Faster automated response triggers',
            'Better backup integration',
            'Enhanced user training'
        ],
        'recommendations': [
            'Implement AI-based anomaly detection',
            'Regular red team exercises',
            'Automated compliance reporting'
        ]
    }

    iris.conduct_post_incident_review(case_id, review_findings)
    post_incident_actions.append(" Conducted post-incident review")
    print("   Conducted post-incident review")

except Exception as e:
    post_incident_actions.append(f" Failed to conduct review: {str(e)}")
    print(f"   Failed to conduct review: {str(e)}")

# Close the incident case
print("\n Closing incident case...")
try:
    closure_data = {
        'closure_reason': 'Incident resolved - threats eradicated, systems recovered',
        'closure_timestamp': datetime.now().isoformat(),
        'final_status': 'CLOSED',
        'follow_up_required': False,
        'reopen_criteria': 'Detection of similar ransomware activity'
    }

    iris.close_case(case_id, closure_data)
    post_incident_actions.append(" Closed incident case")
    print("   Closed incident case")

except Exception as e:
    post_incident_actions.append(f" Failed to close case: {str(e)}")
    print(f"   Failed to close case: {str(e)}")

# Final summary
print(f"\n Post-Incident Summary:")
print(f"  Case ID: {case_id}")
print(f"  Actions Completed: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Preventive Measures: {len(preventive_measures)}")
print(f"  Lessons Learned: {len(lessons_learned)}")

print("\n Incident Response Complete")
print("=" * 60)
print("Ransomware incident successfully contained and resolved.")
print("Systems recovered, monitoring enhanced, and preventive measures implemented.")
print("=" * 60)


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
